# 📄 논문 번역·요약·Q&A 챗봇 — 실행 파이프라인

PDF 논문을 업로드하면 **한국어 요약 · 영→한 번역 · 멀티턴 질의응답**을 제공하는 서비스입니다.
단일 인스트럭트 LLM(`Qwen2.5-Instruct`) 하나로 세 작업을 모두 처리합니다.

```
[Streamlit UI]  PDF 업로드 + 채팅
      │  HTTP (JSON / 파일 + API Key)
      ▼
[FastAPI]  인증 → 텍스트 추출 → 청크 분할 → (요약/번역/Q&A 프롬프트) → 생성
      ▼
[Transformers]  Qwen2.5-Instruct  (CPU: 0.5B / GPU: 1.5B 자동 선택)
```

**이 노트북을 위에서부터 실행하면**: 코드 파일 생성 → 서버 기동 → API 파이프라인 테스트 → UI 실행 순으로 전체 흐름을 확인할 수 있습니다.

| 단계 | 재사용 기술 |
|------|------------|
| PDF 업로드·크기 제한 | Day 6 |
| 번역/요약/Q&A 생성, 멀티턴 | Day 7 |
| API Key 인증 | Day 6 |
| 비동기 추론(run_in_executor) | Day 3 |
| 채팅 UI + 파일 업로드 | Day 4/7 |


## 0. 서버 실행 도우미 — 맨 처음 한 번 실행

노트북 안에서 uvicorn 서버를 띄우고 멈추는 함수를 정의합니다.
(같은 포트에 다른 서버가 있으면 충돌 없이 안내만 하고 멈춥니다.)

In [ ]:
import os, sys, asyncio, threading, time, socket, contextlib
import uvicorn

# app/ 가 있는 위치로 작업 디렉터리를 맞춥니다.
if not os.path.isdir('app') and os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
for _d in ('app', 'models', 'data', 'frontend'):
    os.makedirs(_d, exist_ok=True)

_SERVERS = {}

def _port_open(host, port):
    with contextlib.closing(socket.socket()) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def stop_server(port=8000):
    """이 노트북이 띄운 서버를 멈춥니다."""
    entry = _SERVERS.pop(port, None)
    if not entry:
        print(f'포트 {port}: 이 노트북이 띄운 서버가 없습니다.')
        return
    server, thread = entry
    server.should_exit = True
    for _ in range(50):
        if not thread.is_alive():
            break
        time.sleep(0.1)
    print(f'포트 {port} 서버 종료됨.')

def serve_in_thread(app, host='127.0.0.1', port=8000, log_level='warning'):
    """백그라운드 스레드에서 uvicorn 서버를 띄웁니다. app: 'app.paper_api:app' 같은 경로."""
    stop_server(port)
    if _port_open(host, port):
        print(f'⚠️ 포트 {port}를 다른 프로세스가 사용 중입니다. 그 서버를 끄고 다시 실행하세요.')
        return None
    if isinstance(app, str):
        sys.modules.pop(app.split(':')[0], None)  # 파일을 다시 저장한 경우 최신 내용 반영
    config = uvicorn.Config(app, host=host, port=port, log_level=log_level, loop='asyncio')
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    def _run():
        loop = asyncio.SelectorEventLoop() if sys.platform == 'win32' else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())
    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)
    for i in range(600):                      # 모델 로드 때문에 최대 5분 대기
        if _port_open(host, port):
            print(f'서버 실행됨: http://{host}:{port}')
            return server
        if not thread.is_alive():
            print('서버 스레드가 종료됐습니다. 위 로그를 확인하세요.')
            return server
        if i > 0 and i % 20 == 0:
            print(f'  ... 모델 로드 중 ({i//2}초 경과)')
        time.sleep(0.5)
    print('5분 내에 시작되지 않았습니다.')
    return server

print('서버 도우미 준비 완료 (serve_in_thread, stop_server)')

## 1. 환경 확인

필요 패키지: `torch`, `transformers`, `pypdf`, `fastapi`, `uvicorn`, `streamlit`.
없으면 아래 주석을 풀어 설치하세요.

In [ ]:
# !pip install torch transformers pypdf fastapi uvicorn streamlit -q
import torch, transformers, pypdf, fastapi, streamlit
print('torch        :', torch.__version__)
print('transformers :', transformers.__version__)
print('GPU 사용 가능 :', torch.cuda.is_available(), '(없으면 CPU용 0.5B 모델 자동 선택)')

## 2. 서비스 코드 작성

각 셀을 실행하면 `app/`·`frontend/`에 코드 파일이 생성됩니다.
(이미 파일이 있어도 동일 내용으로 다시 씁니다.)

| 파일 | 역할 |
|------|------|
| `app/paper_utils.py` | PDF 텍스트 추출 + 청크 분할 |
| `app/paper_schemas.py` | Pydantic 요청/응답 스키마 |
| `app/paper_model.py` | 단일 LLM 요약·번역·Q&A |
| `app/paper_api.py` | FastAPI 서버 (인증·업로드·비동기) |
| `frontend/app_paper.py` | Streamlit UI |

> 인증/로깅/에러/미들웨어(`app/auth.py`, `logger_config.py`, `error_handlers.py`, `middleware.py`)는
> Day 3/6에서 만든 것을 **그대로 재사용**합니다.

### `app/paper_utils.py` — PDF 추출 + 청크

In [ ]:
%%writefile app/paper_utils.py
"""
Day 8 - 논문 PDF 텍스트 추출 + 청크 분할 유틸
(Day 6의 파일 업로드/안전장치 개념을 텍스트 추출로 확장)
"""
import io
import re
from pypdf import PdfReader


def extract_text_from_pdf(file_bytes: bytes, max_pages: int = 30) -> str:
    """PDF 바이트에서 텍스트를 추출합니다.

    max_pages: 너무 긴 논문에서 앞쪽 N페이지만 읽어 CPU 부담을 줄입니다.
    """
    reader = PdfReader(io.BytesIO(file_bytes))
    pages = reader.pages[:max_pages]
    texts = []
    for page in pages:
        t = page.extract_text() or ""
        texts.append(t)
    text = "\n".join(texts)
    return _clean(text)


def _clean(text: str) -> str:
    """추출 과정에서 생긴 과도한 공백/줄바꿈을 정리합니다."""
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_text(text: str, max_chars: int = 1500, overlap: int = 100) -> list[str]:
    """긴 텍스트를 문단 경계 기준으로 청크로 나눕니다.

    소형 모델은 컨텍스트가 짧으므로, 한 번에 처리할 만한 크기로 자릅니다.
    overlap: 청크 경계에서 문맥이 끊기지 않도록 약간 겹쳐줍니다.
    """
    if not text:
        return []
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks: list[str] = []
    buf = ""
    for p in paragraphs:
        if len(buf) + len(p) + 2 <= max_chars:
            buf = f"{buf}\n\n{p}" if buf else p
        else:
            if buf:
                chunks.append(buf)
            # 문단 하나가 너무 길면 강제로 자른다
            if len(p) > max_chars:
                for i in range(0, len(p), max_chars - overlap):
                    chunks.append(p[i:i + max_chars])
                buf = ""
            else:
                buf = p
    if buf:
        chunks.append(buf)
    return chunks

### `app/paper_schemas.py` — 스키마

In [ ]:
%%writefile app/paper_schemas.py
"""
Day 8 - 논문 챗봇 API 스키마
(Day 5/7에서 배운 Pydantic 스키마 설계)
"""
from pydantic import BaseModel, Field
from typing import Optional


class UploadResponse(BaseModel):
    """PDF 업로드 결과"""
    doc_id: str = Field(description="업로드된 문서 식별자")
    filename: str = Field(description="원본 파일명")
    n_chars: int = Field(description="추출된 글자 수")
    n_chunks: int = Field(description="분할된 청크 수")
    preview: str = Field(description="앞부분 미리보기")


class SummarizeRequest(BaseModel):
    """요약 요청"""
    doc_id: str = Field(..., description="업로드한 문서 ID")
    max_chunks: int = Field(default=6, ge=1, le=20,
                            description="요약에 사용할 최대 청크 수 (CPU 속도 조절)")


class TranslateRequest(BaseModel):
    """번역 요청 (영→한 기본)"""
    doc_id: str = Field(..., description="업로드한 문서 ID")
    max_chunks: int = Field(default=3, ge=1, le=20,
                            description="번역할 최대 청크 수")


class Message(BaseModel):
    """단일 대화 메시지"""
    role: str = Field(..., description="역할: 'user' 또는 'bot'")
    content: str = Field(..., min_length=1, description="메시지 내용")


class ChatRequest(BaseModel):
    """논문 내용 기반 멀티턴 Q&A 요청"""
    doc_id: str = Field(..., description="업로드한 문서 ID")
    messages: list[Message] = Field(..., min_length=1,
                                    description="대화 기록 (마지막이 현재 질문)")
    max_new_tokens: int = Field(default=256, ge=10, le=1024)
    temperature: float = Field(default=0.7, gt=0.0, le=2.0)


class TextResponse(BaseModel):
    """요약/번역/답변 공통 응답"""
    success: bool = Field(description="성공 여부")
    result: str = Field(description="생성된 텍스트")
    doc_id: str = Field(description="대상 문서 ID")
    model_name: str = Field(description="사용된 모델")
    user: Optional[str] = Field(default=None, description="인증된 사용자")

### `app/paper_model.py` — 모델 (요약·번역·Q&A)

In [ ]:
%%writefile app/paper_model.py
"""
Day 8 - 논문 챗봇 모델 (단일 인스트럭트 LLM으로 번역·요약·Q&A 모두 처리)
Day 7의 ChatbotModel을 확장 — 같은 모델에 '역할(system) + 작업 프롬프트'만 바꿔 재사용한다.
"""
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


class PaperBot:
    """하나의 LLM으로 논문 요약/번역/질의응답을 수행합니다."""

    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"):
        if torch.cuda.is_available():
            self.device = "cuda"
        elif torch.backends.mps.is_available():
            self.device = "mps"
        else:
            self.device = "cpu"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        dtype = torch.float16 if self.device in ("cuda", "mps") else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype)
        self.model = self.model.to(self.device)
        self.model.eval()
        self.model_name = model_name

    # ---- 핵심 생성기 (모든 작업이 이걸 공유) ----
    def _generate(self, chat: list[dict], max_new_tokens: int = 256,
                  temperature: float = 0.7) -> str:
        # transformers 5.x: return_dict=False 로 텐서를 직접 받는다 (안 하면 BatchEncoding)
        input_ids = self.tokenizer.apply_chat_template(
            chat, add_generation_prompt=True, return_tensors="pt", return_dict=False
        ).to(self.device)

        max_length = getattr(self.model.config, "max_position_embeddings", 2048)
        if input_ids.shape[1] > max_length - max_new_tokens:
            # 입력이 너무 길면 앞부분을 잘라 모델 한계를 넘지 않게 한다
            input_ids = input_ids[:, -(max_length - max_new_tokens):]

        with torch.no_grad():
            output_ids = self.model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=0.9,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        return self.tokenizer.decode(
            output_ids[0][input_ids.shape[1]:], skip_special_tokens=True
        ).strip()

    # ---- 요약: 청크별 요약 → 합쳐서 최종 요약 (map-reduce) ----
    def summarize(self, chunks: list[str], max_chunks: int = 6) -> str:
        used = chunks[:max_chunks]
        partial = []
        for i, ch in enumerate(used, 1):
            chat = [
                {"role": "system",
                 "content": "너는 학술 논문을 읽고 핵심을 한국어로 정리하는 전문가야."},
                {"role": "user",
                 "content": f"다음 논문 일부를 한국어로 3문장 이내로 요약해줘.\n\n{ch}"},
            ]
            partial.append(f"[{i}] " + self._generate(chat, max_new_tokens=200, temperature=0.5))

        joined = "\n".join(partial)
        chat = [
            {"role": "system",
             "content": "너는 부분 요약들을 통합해 하나의 매끄러운 한국어 요약으로 만드는 전문가야."},
            {"role": "user",
             "content": f"다음 부분 요약들을 합쳐 논문 전체를 한국어로 5~7문장으로 요약해줘.\n\n{joined}"},
        ]
        return self._generate(chat, max_new_tokens=400, temperature=0.5)

    # ---- 번역: 청크별 영→한 번역 ----
    def translate(self, chunks: list[str], max_chunks: int = 3) -> str:
        out = []
        for ch in chunks[:max_chunks]:
            chat = [
                {"role": "system",
                 "content": "너는 영어 학술 텍스트를 자연스러운 한국어로 번역하는 전문 번역가야. "
                            "전문 용어는 정확히 옮기고, 의미를 보존해."},
                {"role": "user", "content": f"다음 영어 텍스트를 한국어로 번역해줘.\n\n{ch}"},
            ]
            out.append(self._generate(chat, max_new_tokens=512, temperature=0.3))
        return "\n\n".join(out)

    # ---- Q&A: 논문 내용을 컨텍스트로 멀티턴 답변 ----
    def answer(self, context: str, messages: list[dict],
               max_new_tokens: int = 256, temperature: float = 0.7) -> str:
        chat = [
            {"role": "system",
             "content": "너는 주어진 논문 내용에 근거해 한국어로 답하는 助手야. "
                        "논문에 없는 내용은 추측하지 말고 모른다고 말해."},
            {"role": "user", "content": f"[논문 내용]\n{context}\n\n위 내용을 참고해 대화에 답해줘."},
            {"role": "assistant", "content": "네, 논문 내용을 바탕으로 답변하겠습니다."},
        ]
        for m in messages:
            role = "user" if m["role"] == "user" else "assistant"
            chat.append({"role": role, "content": m["content"]})
        return self._generate(chat, max_new_tokens=max_new_tokens, temperature=temperature)

### `app/paper_api.py` — FastAPI 서버

In [ ]:
%%writefile app/paper_api.py
"""
Day 8 - 논문 번역·요약·Q&A 챗봇 FastAPI 서버
재사용: auth(Day6) / logger·error·middleware(Day3) / 비동기 추론(Day3) / 업로드(Day6)
"""
import asyncio
import uuid
from contextlib import asynccontextmanager
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI, Depends, HTTPException, UploadFile, File

from app.paper_schemas import (
    UploadResponse, SummarizeRequest, TranslateRequest, ChatRequest, TextResponse,
)
from app.paper_model import PaperBot
from app.paper_utils import extract_text_from_pdf, chunk_text
from app.auth import verify_api_key
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware

logger = setup_logger("paper_api")
inference_executor = ThreadPoolExecutor(max_workers=2, thread_name_prefix="paper")

MAX_PDF_BYTES = 10 * 1024 * 1024          # 10MB 업로드 제한 (Day 6 안전장치)
CONTEXT_CHARS = 3000                       # Q&A에 넣을 논문 컨텍스트 길이

bot: PaperBot | None = None
# 업로드된 문서를 메모리에 보관 (간단한 데모용 — 운영에선 DB/스토리지 권장)
DOCS: dict[str, dict] = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    global bot
    import torch
    use_gpu = torch.cuda.is_available() or torch.backends.mps.is_available()
    model_name = "Qwen/Qwen2.5-1.5B-Instruct" if use_gpu else "Qwen/Qwen2.5-0.5B-Instruct"
    logger.info(f"논문 챗봇 모델 로드 중: {model_name}")
    bot = PaperBot(model_name)
    logger.info("모델 로드 완료")
    yield


app = FastAPI(
    title="Paper Chatbot API",
    description="논문 PDF 번역·요약·질의응답 API (인증 필요)",
    version="1.0.0",
    lifespan=lifespan,
)
app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)


def _get_doc(doc_id: str) -> dict:
    doc = DOCS.get(doc_id)
    if doc is None:
        raise HTTPException(status_code=404, detail="문서를 찾을 수 없습니다. 먼저 업로드하세요.")
    return doc


@app.get("/health", tags=["System"])
async def health_check():
    return {"status": "healthy" if bot else "loading",
            "model": bot.model_name if bot else None,
            "docs": len(DOCS)}


@app.post("/upload", response_model=UploadResponse, tags=["Paper"])
async def upload(file: UploadFile = File(...), user: str = Depends(verify_api_key)):
    """PDF 논문을 업로드해 텍스트를 추출하고 doc_id를 발급합니다."""
    if file.content_type not in ("application/pdf", "application/octet-stream"):
        raise HTTPException(status_code=400, detail="PDF 파일만 업로드할 수 있습니다.")
    data = await file.read()
    if len(data) > MAX_PDF_BYTES:
        raise HTTPException(status_code=413, detail="파일이 너무 큽니다 (최대 10MB).")

    try:
        text = extract_text_from_pdf(data)
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"PDF 텍스트 추출 실패: {e}")
    if not text:
        raise HTTPException(status_code=400, detail="텍스트를 추출하지 못했습니다 (스캔 PDF일 수 있음).")

    chunks = chunk_text(text)
    doc_id = uuid.uuid4().hex[:8]
    DOCS[doc_id] = {"filename": file.filename, "text": text, "chunks": chunks}
    logger.info(f"업로드 — 사용자:{user} 파일:{file.filename} 청크:{len(chunks)}")
    return UploadResponse(doc_id=doc_id, filename=file.filename, n_chars=len(text),
                          n_chunks=len(chunks), preview=text[:300])


async def _run(fn, *args):
    if bot is None:
        raise HTTPException(status_code=503, detail="모델이 아직 로드되지 않았습니다.")
    try:
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(inference_executor, fn, *args)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"추론 실패: {e}")


@app.post("/summarize", response_model=TextResponse, tags=["Paper"])
async def summarize(req: SummarizeRequest, user: str = Depends(verify_api_key)):
    doc = _get_doc(req.doc_id)
    result = await _run(bot.summarize, doc["chunks"], req.max_chunks)
    return TextResponse(success=True, result=result, doc_id=req.doc_id,
                        model_name=bot.model_name, user=user)


@app.post("/translate", response_model=TextResponse, tags=["Paper"])
async def translate(req: TranslateRequest, user: str = Depends(verify_api_key)):
    doc = _get_doc(req.doc_id)
    result = await _run(bot.translate, doc["chunks"], req.max_chunks)
    return TextResponse(success=True, result=result, doc_id=req.doc_id,
                        model_name=bot.model_name, user=user)


@app.post("/chat", response_model=TextResponse, tags=["Paper"])
async def chat(req: ChatRequest, user: str = Depends(verify_api_key)):
    doc = _get_doc(req.doc_id)
    context = doc["text"][:CONTEXT_CHARS]
    msgs = [m.model_dump() for m in req.messages]
    result = await _run(bot.answer, context, msgs, req.max_new_tokens, req.temperature)
    return TextResponse(success=True, result=result, doc_id=req.doc_id,
                        model_name=bot.model_name, user=user)

### `frontend/app_paper.py` — Streamlit UI

In [ ]:
%%writefile frontend/app_paper.py
"""
Day 8 - 논문 번역·요약·Q&A 챗봇 대시보드 (Streamlit)
재사용: Day 4 파일 업로드/레이아웃 + Day 7 채팅 UI(st.chat_message/st.chat_input)
"""
import streamlit as st
import requests

st.set_page_config(page_title="논문 챗봇", page_icon="📄", layout="centered")
API_BASE = "http://localhost:8000"


def api_post(path, api_key, json=None, files=None, timeout=180):
    try:
        resp = requests.post(f"{API_BASE}{path}",
                             json=json, files=files,
                             headers={"X-API-Key": api_key}, timeout=timeout)
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.ConnectionError:
        st.error("🔌 서버에 연결할 수 없습니다.")
    except requests.exceptions.HTTPError as e:
        code = e.response.status_code
        if code == 401:
            st.error("🔑 인증 실패. API Key를 확인하세요.")
        else:
            detail = e.response.json().get("detail", "")
            st.error(f"❌ 서버 에러 (HTTP {code}) {detail}")
    except Exception as e:
        st.error(f"❌ 오류: {type(e).__name__}")
    return None


# ===== 사이드바 =====
with st.sidebar:
    st.header("⚙️ 설정")
    api_key = st.text_input("API Key", value="test-key-001", type="password")
    st.divider()

    pdf = st.file_uploader("논문 PDF 업로드", type=["pdf"])
    if st.button("📤 업로드 & 분석", disabled=pdf is None):
        with st.spinner("PDF 텍스트 추출 중..."):
            files = {"file": (pdf.name, pdf.getvalue(), "application/pdf")}
            res = api_post("/upload", api_key, files=files)
        if res:
            st.session_state["doc_id"] = res["doc_id"]
            st.session_state["doc_name"] = res["filename"]
            st.session_state["messages"] = []
            st.success(f"✅ 업로드 완료: {res['n_chunks']}개 청크 / {res['n_chars']:,}자")

    if st.session_state.get("doc_id"):
        st.caption(f"📄 현재 문서: {st.session_state['doc_name']} (id={st.session_state['doc_id']})")
    st.divider()

    try:
        h = requests.get(f"{API_BASE}/health", timeout=3).json()
        st.success(f"🟢 서버 연결됨 · 모델 {h.get('model','')}") if h.get("status") == "healthy" \
            else st.warning("🟡 모델 로딩 중...")
    except Exception:
        st.error("🔴 서버 연결 실패")


# ===== 상태 초기화 =====
st.session_state.setdefault("messages", [])

st.title("📄 논문 번역·요약 챗봇")

doc_id = st.session_state.get("doc_id")
if not doc_id:
    st.info("왼쪽 사이드바에서 논문 PDF를 업로드하세요.")
else:
    # ---- 요약 / 번역 버튼 ----
    c1, c2 = st.columns(2)
    if c1.button("📝 전체 요약"):
        with st.spinner("요약 생성 중... (CPU라 다소 걸립니다)"):
            res = api_post("/summarize", api_key, json={"doc_id": doc_id, "max_chunks": 6})
        if res:
            st.subheader("요약 결과")
            st.write(res["result"])
    if c2.button("🌐 영→한 번역 (앞부분)"):
        with st.spinner("번역 중..."):
            res = api_post("/translate", api_key, json={"doc_id": doc_id, "max_chunks": 3})
        if res:
            st.subheader("번역 결과")
            st.write(res["result"])

    st.divider()
    st.caption("논문 내용에 대해 질문해 보세요 (멀티턴).")

    # ---- 대화 기록 표시 ----
    for m in st.session_state["messages"]:
        with st.chat_message(m["role"]):
            st.write(m["content"])

    # ---- 질문 입력 ----
    if q := st.chat_input("논문에 대해 질문하기..."):
        st.session_state["messages"].append({"role": "user", "content": q})
        with st.chat_message("user"):
            st.write(q)
        with st.chat_message("assistant"):
            with st.spinner("답변 생성 중..."):
                api_msgs = [{"role": "user" if m["role"] == "user" else "bot",
                             "content": m["content"]} for m in st.session_state["messages"]]
                res = api_post("/chat", api_key,
                               json={"doc_id": doc_id, "messages": api_msgs})
            if res:
                st.write(res["result"])
                st.session_state["messages"].append(
                    {"role": "assistant", "content": res["result"]})

## 3. (선택) 모델 단독 테스트

서버 없이 `PaperBot`을 직접 불러 생성 경로를 빠르게 확인합니다. (모델 다운로드/로드에 시간이 걸립니다.)

In [ ]:
from app.paper_model import PaperBot

_bot = PaperBot()   # CPU면 Qwen2.5-0.5B-Instruct 자동 선택
print('device =', _bot.device, '| model =', _bot.model_name)
_eng = ("Transformers are deep learning models that use self-attention. "
        "They have become the standard for natural language processing tasks.")
print('요약:', _bot.summarize([_eng], max_chunks=1))

## 4. 백엔드 서버 실행

`app.paper_api:app`을 8000 포트에 띄웁니다. (이미 다른 서버가 8000을 쓰면 끄고 다시 실행하세요.)

In [ ]:
server = serve_in_thread("app.paper_api:app", host="127.0.0.1", port=8000)

## 5. API 파이프라인 테스트

업로드 → 요약 → 번역 → Q&A → 인증 실패 순으로 전체 흐름을 검증합니다.

### 5.0 샘플 PDF 만들기 (직접 업로드할 PDF가 있으면 건너뛰세요)

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

_text = (
    "Title: Attention Is All You Need (Sample)\n\n"
    "Abstract. We propose the Transformer, a network architecture based solely on "
    "attention mechanisms, dispensing with recurrence and convolutions entirely.\n\n"
    "1. Introduction. Recurrent models factor computation along symbol positions, "
    "which precludes parallelization within training examples.\n\n"
    "2. Method. The Transformer uses stacked self-attention and fully connected layers. "
    "Multi-head attention attends to different representation subspaces.\n\n"
    "3. Results. The model achieves state-of-the-art BLEU scores while being more "
    "parallelizable and requiring less time to train."
)
with PdfPages("test_paper.pdf") as pdf:
    fig = plt.figure(figsize=(8.27, 11.69))
    fig.text(0.07, 0.95, _text, va="top", ha="left", fontsize=11, wrap=True)
    plt.axis("off"); pdf.savefig(fig); plt.close(fig)
print("test_paper.pdf 생성 완료")

### 5.1 PDF 업로드 → doc_id 발급

In [ ]:
import requests
BASE = "http://localhost:8000"
KEY  = {"X-API-Key": "test-key-001"}

with open("test_paper.pdf", "rb") as fp:
    r = requests.post(f"{BASE}/upload", headers=KEY,
                      files={"file": ("test_paper.pdf", fp, "application/pdf")}, timeout=60)
up = r.json()
doc_id = up["doc_id"]
print("status:", r.status_code)
print(f"doc_id={doc_id}  글자수={up['n_chars']}  청크={up['n_chunks']}")
print("미리보기:", up["preview"][:150])

### 5.2 요약

In [ ]:
r = requests.post(f"{BASE}/summarize", headers=KEY,
                  json={"doc_id": doc_id, "max_chunks": 4}, timeout=300)
print(r.status_code)
print(r.json()["result"])

### 5.3 영→한 번역 (앞부분)

In [ ]:
r = requests.post(f"{BASE}/translate", headers=KEY,
                  json={"doc_id": doc_id, "max_chunks": 2}, timeout=300)
print(r.status_code)
print(r.json()["result"])

### 5.4 멀티턴 Q&A

In [ ]:
r = requests.post(f"{BASE}/chat", headers=KEY,
                  json={"doc_id": doc_id,
                        "messages": [{"role": "user", "content": "이 논문이 제안하는 핵심 아키텍처는?"}],
                        "max_new_tokens": 150}, timeout=300)
print(r.status_code)
print(r.json()["result"])

### 5.5 인증 실패 확인 (API Key 없이 → 401)

In [ ]:
r = requests.post(f"{BASE}/summarize", json={"doc_id": doc_id}, timeout=30)
print("status:", r.status_code, "(401 이어야 정상)")

## 6. Streamlit UI 실행

별도 프로세스로 채팅 UI를 띄웁니다. 실행 후 브라우저에서 **http://localhost:8501** 접속.
(백엔드 서버가 8000에 떠 있어야 합니다.)

In [ ]:
import subprocess, sys
ui = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "frontend/app_paper.py",
     "--server.port", "8501", "--server.headless", "true"]
)
print("✅ 프론트엔드: http://localhost:8501  (PID", ui.pid, ")")

## 7. 정리 / 종료

실습이 끝나면 서버와 UI를 정리합니다.

In [ ]:
stop_server(8000)
try:
    ui.terminate(); print("Streamlit 종료")
except Exception:
    pass

## 📌 실행 파이프라인 요약

```
0. 서버 도우미 정의
1. 환경 확인
2. 코드 작성 (app/ + frontend/)        ← %%writefile
3. (선택) 모델 단독 테스트
4. 백엔드 서버 실행                      serve_in_thread("app.paper_api:app")
5. API 테스트  업로드→요약→번역→Q&A→인증
6. Streamlit UI 실행                     http://localhost:8501
7. 종료                                  stop_server(8000)
```

### 회고 포인트
- 단일 LLM을 **프롬프트만 바꿔** 번역·요약·Q&A에 재사용 → Day 7 코드의 자연스러운 확장
- CPU/소형 모델 한계: 번역 환각, 긴 논문 처리 속도 → `max_chunks`로 범위 제한
- 개선 방향: 번역 전용 모델(NLLB) 추가, 임베딩 기반 검색(RAG)으로 Q&A 정확도 향상
